In [1]:
import torch
from torch import nn
from torch.distributions import Categorical
from environment import Environment
import numpy as np
import torch.nn.functional as F
import torch.optim as optim
from itertools import count

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
env = Environment()

In [4]:
num_actions = 50
action_range = np.linspace(-2, 2, num_actions)

In [5]:
class Critic(nn.Module):
    def __init__(self, in_features):
        super(Critic, self).__init__()
        self.fc1 = nn.Linear(in_features, 128) 
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 1) 
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [6]:
def train_only_critic():
    R = 0
    returns = []
    
    for r in rewards[::-1]:
        R = r + 0.99 * R # Gamma is 0.99
        returns.insert(0, R)

    returns = torch.tensor(returns, device=device)
    returns = (returns - returns.mean()) / (returns.std() + eps)

    old_probs, state_values, states, actions = zip(*saved_actions)
    state_values = torch.stack(state_values).to(device)

    critic_loss = F.smooth_l1_loss(state_values.squeeze(), returns)
    critic_optimizer.zero_grad()
    critic_loss.backward(retain_graph=False)
    critic_optimizer.step()

    print(critic_loss.item())

    del rewards[:]
    del saved_actions[:]

In [7]:
def finish_episode():
    # Calculating losses and performing backprop
    R = 0
    returns = []
    epsilon = 0.2
    num_epochs = 1

    for r in rewards[::-1]:
        R = r + 0.9 * R # Gamma is 0.99
        returns.insert(0, R)

    returns = torch.tensor(returns, device=device)
    returns = (returns - returns.mean()) / (returns.std() + eps)

    old_probs, state_values, states, actions = zip(*saved_actions)

    old_probs = torch.stack(old_probs).to(device)
    state_values = torch.stack(state_values).to(device)
    states = torch.stack(states).to(device)
    actions = torch.stack(actions).to(device)

    advantages = returns - state_values.squeeze()

    for epoch in range(num_epochs):

        new_probs = actor(states).gather(1, actions).squeeze()

        ratios = new_probs / old_probs

        surr1 = ratios * advantages
        surr2 = torch.clamp(ratios, 1 - epsilon, 1 + epsilon) * advantages

        #actor_loss = -torch.min(surr1, surr2).mean()
        actor_loss = -surr1.mean()

        actor_optimizer.zero_grad()
        actor_loss.backward(retain_graph=True)
        actor_optimizer.step()

        if epoch == num_epochs - 1:
            critic_loss = F.smooth_l1_loss(state_values.squeeze(), returns)
            critic_optimizer.zero_grad()
            critic_loss.backward(retain_graph=False)
            critic_optimizer.step()

    del rewards[:]
    del saved_actions[:]

In [8]:
def select_action(state_and_info):
    logits = actor(state_and_info.unsqueeze(0))
    probs = F.softmax(logits, dim=-1)
    distribution = Categorical(probs)
    action = distribution.sample()

    state_value = critic(state_and_info.unsqueeze(0))

    saved_actions.append((probs[0][action.item()].detach(), state_value, state_and_info, action.detach()))

    action_item = action.item()

    return action_item

In [9]:
def train(train_critic_only=False, n_episodes = 100):
    best_cost = 10000

    for i_episode in range(n_episodes):
        state = env.reset()

        integral = 0
        prev_error = 0
        prev_action = 0
        prev_derivative = 0

        while True:
            error = state[0] - state[1]
            integral += error
            derivative = error - prev_error

            action_idx = select_action(torch.tensor([*state, error, prev_error, integral, derivative, prev_derivative, prev_action], dtype=torch.float32, device=device))
            action = action_range[action_idx]

            next_state, cost, done = env.step(action)

            rewards.append(-cost[2])

            state = next_state

            prev_error = error
            prev_derivative = derivative
            prev_action = action

            if done:
                if train_critic_only:
                    train_only_critic()
                else:
                    total_cost = env.get_total_cost()
                    if total_cost[2] < best_cost: best_cost = total_cost[2]
                    print(f'Episode {i_episode} Total Cost:', total_cost)
                    finish_episode()
                break

    return best_cost

In [10]:
actor = torch.load('./models/mlp_pid.pth') # model takes: 'target_lataccel', 'current_lataccel', 'vEgo', 'aEgo', 'roll', 'error', 'prev_error', 'integral', 'derivative', 'prev_derivative', 'prev_action'
actor.to(device)
actor.eval()
actor_optimizer = optim.Adam(actor.parameters(), lr=5e-5)
saved_actions = []
rewards = []

critic = Critic(in_features=11).to(device)
critic_optimizer = optim.Adam(critic.parameters(), lr=3e-2)
eps = np.finfo(np.float32).eps.item()

In [11]:
train(train_critic_only=True, n_episodes=20)
best_cost = train(n_episodes=500)

1.752394613265991
25.015218688964843
8.433495700836183
1.6862048292060055
1.423785561850349
0.6167274510011107
0.4871074851855367
0.6057721236226551
0.5518217911252498
0.484186535881768
0.3964242368370713
0.3811177817194078
0.4144350180769603
0.44886929513475005
0.3774958043997209
0.38521715410949625
0.4232054734194686
0.3641821888409345
0.36031032709818195
0.3791036408742106
Episode 0 Total Cost: [  3.75786726  92.56288118 111.35221747]
Episode 1 Total Cost: [  3.43031706  85.33218402 102.48376931]
Episode 2 Total Cost: [ 3.63043338 81.0465119  99.19867877]
Episode 3 Total Cost: [ 3.94054685 73.94334769 93.64608195]
Episode 4 Total Cost: [ 3.44603499 82.21279692 99.44297189]
Episode 5 Total Cost: [  3.41598902  82.98833515 100.06828025]
Episode 6 Total Cost: [ 3.57163868 73.57847132 91.43666473]
Episode 7 Total Cost: [  3.53770289  86.60559864 104.29411311]
Episode 8 Total Cost: [ 3.73822516 69.95593225 88.64705805]
Episode 9 Total Cost: [  3.93885172  85.23460982 104.92886841]
Episod